In [1]:
import numpy as np
import pandas as pd

from lazypredict.Supervised import LazyRegressor
from tpot import TPOTRegressor
from flaml import AutoML
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, RegressorMixin

In [2]:
# -----------------------------
# 1) Chargement des données
# -----------------------------
data = pd.read_csv('../../data/dataset_pv_variability_paca_2020_2024.csv', index_col=0).dropna()

# Variables retardées (mémoire du système)

# Production passée (inertie système)
data["tch_lag_1"] = data["tch"].shift(1)
data["tch_lag_2"] = data["tch"].shift(2)
data["tch_lag_3"] = data["tch"].shift(3)

# Variations irradiance passées
data["dghi_lag_1"] = data["dghi_dt"].shift(1)
data["dghi_lag_2"] = data["dghi_dt"].shift(2)
data["dghi_lag_3"] = data["dghi_dt"].shift(3)

# Variations production normalisée passées
data["dtch_lag_1"] = data["target_variability"].shift(1)
data["dtch_lag_2"] = data["target_variability"].shift(2)
data["dtch_lag_3"] = data["target_variability"].shift(3)

# Variations nuages passées
data["dcloud_lag_1"] = data["dcloud_dt"].shift(1)
data["dcloud_lag_2"] = data["dcloud_dt"].shift(2)

# Nuit / jour
data["is_night"] = (data["ghi_region"] <= 0).astype(int)

data["ghi_lag_3"] = data["ghi_region"].shift(3)

# Nettoyage (suppression lignes avec NaN dus aux lags)
df = data.dropna().copy()

# -----------------------------
# Split temporel train/test
# -----------------------------
train = df.loc[df['year'] <= 2022].copy()
test = df.loc[df['year'] == 2023].copy()

# Liste des features
features = ["sin_hour", "cos_hour", "sin_doy", "cos_doy","dghi_dt", "dcloud_dt", 
            "cloud_lag_1", "cloud_lag_2", "dcloud_lag_1", "dcloud_lag_2", "cloud_cover_region",
            "is_night", "tch_lag_1", "tch_lag_2", "tch_lag_3", "ghi_lag_1", "ghi_lag_2", "ghi_lag_3",
            "dghi_lag_1", "dghi_lag_2", "dghi_lag_3", "dtch_lag_1", "dtch_lag_2", "dtch_lag_3",
            "temperature_region", 'wind_speed_region', 'humidity_region', 'toa_irradiance_region', "season",
            'solar_azimuth_region', 'solar_altitude_region', 'temperature_region',
            'ghi_region', 'bhi_region', 'dhi_region', 'dni_region',
            'wind_speed_region', 'cloud_cover_region', 'humidity_region',
            'clear_sky_ghi_region', 'clear_sky_bhi_region', 'clear_sky_dhi_region',
            'clear_sky_dni_region', 'month'
]

# Supprimer des doublons éventuels
features = list(dict.fromkeys(features))

# Vérifier l’existence des features dans les colonnes de df
missing = [c for c in features if c not in df.columns]
if missing:
    raise ValueError(f"Colonnes manquantes dans le DataFrame df: {missing}")
    
# Matrices train/test
X_train = train[features]
y_train = train["target_variability"]

X_test = test[features]
y_test = test["target_variability"]

# ------------------------------
# Contrôle automatique NaN / inf
# ------------------------------
def sanity_check(X, name="X"):
    if np.any(~np.isfinite(X.select_dtypes(include=["number"]).to_numpy())):
        raise ValueError(f"{name} contient des valeurs NaN dans les colonnes numériques.")
    # For categorical columns: ensure no NaN
    cat = X.select_dtypes(exclude=["number"])
    if cat.shape[1] > 0 and cat.isna().any().any():
        raise ValueError(f"{name} contient des valeurs NaN dans les colonnes catégorielles.")

sanity_check(X_train, "X_train")
sanity_check(X_test,  "X_test")

# -----------------------------
# Séparation colonnes numériques / catégorielles
# -----------------------------
num_cols = X_train.select_dtypes(include=["number"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

# -----------------------------
# Préprocessing: unscaled vs scaled
# -----------------------------
numeric_scale = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# A) Préprocesseur sans normalisation :
# - Les variables numériques sont conservées telles quelles (pas de scaling)
# - Les variables catégorielles sont transformées en variables binaires (One-Hot Encoding)
preprocess_unscaled = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

# B) Préprocesseur avec normalisation :
# - Les variables numériques sont standardisées (StandardScaler)
# - Les variables catégorielles sont encodées en variables binaires (One-Hot Encoding)
preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_scale, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

In [3]:
# ---------------------------------
# LazyRegressor + MAE + MAE relatif
# ---------------------------------
def run_lazy(preprocess, label):
    # Fit/transform train, transform test
    Xtr = preprocess.fit_transform(X_train)
    Xte = preprocess.transform(X_test)

    # Lancer LazyRegressor + récupérer prédictions
    reg = LazyRegressor(verbose=1, ignore_warnings=True, random_state=42, predictions=True)
    models, preds = reg.fit(Xtr, Xte, y_train, y_test)
    
    # Référence pour MAE relative (%)
    y_mean = float(np.mean(y_test))

    mae_scores = {}
    mae_rel_scores = {}

    for model_name in preds.columns:
        pred = preds[model_name]
        mae = mean_absolute_error(y_test, pred)
        mae_rel = round(mae / y_mean * 100, 2) if y_mean != 0 else np.nan

        mae_scores[model_name] = mae
        mae_rel_scores[model_name] = mae_rel

        # print(f"🔹 [{label}] {model_name}")
        # print(f"MAE = {mae:.6f}")
        # print(f"le modèle se trompe en moyenne de {mae_rel:.6f}% de la valeur moyenne\n")

    # Fusionner scores MAE avec tableau LazyPredict
        mae_df = pd.DataFrame({
        "MAE": pd.Series(mae_scores),
        "MAE_relatif_%": pd.Series(mae_rel_scores)
    })

    out = models.merge(mae_df, left_index=True, right_index=True)
    out["scaling_mode"] = label
    return out

# Runs
results_unscaled = run_lazy(preprocess_unscaled, "unscaled")
results_scaled   = run_lazy(preprocess_scaled, "scaled")

# Combiner les deux modes
results_all = pd.concat([results_unscaled, results_scaled], axis=0)

# Tableau final propre
models_df = pd.DataFrame(results_all)
models_df = models_df.rename({
    "R-Squared":"r2_score", 
    "MAE_relatif_%":"MAE_relative(%)", 
    "Time Taken":"temps_execution"
}, axis=1)
models_df = models_df[["r2_score", "MAE", "RMSE", "MAE_relative(%)", "scaling_mode", "temps_execution"]]

# afficher les 40 meilleurs modèles avec R² > 0, trier par R² décroissant puis MAE croissant
models_df.loc[models_df["r2_score"] > 0].sort_values(["r2_score", "MAE"], ascending=[False, True]).head(40)

  0%|          | 0/42 [00:00<?, ?it/s]

{'Model': 'AdaBoostRegressor', 'R-Squared': 0.5033949181802528, 'Adjusted R-Squared': 0.5021730127946813, 'RMSE': np.float64(2.4192734448165645), 'Time taken': 30.304697275161743}
{'Model': 'BaggingRegressor', 'R-Squared': 0.9192230656413799, 'Adjusted R-Squared': 0.919024312598497, 'RMSE': np.float64(0.9757151809157253), 'Time taken': 31.30304789543152}
{'Model': 'BayesianRidge', 'R-Squared': 0.8680791905685532, 'Adjusted R-Squared': 0.8677545971372445, 'RMSE': np.float64(1.2469126542736264), 'Time taken': 0.5120315551757812}
{'Model': 'DecisionTreeRegressor', 'R-Squared': 0.8552884799309642, 'Adjusted R-Squared': 0.8549324147350974, 'RMSE': np.float64(1.3059631761085755), 'Time taken': 5.1398766040802}
{'Model': 'DummyRegressor', 'R-Squared': -0.0019438939815170642, 'Adjusted R-Squared': -0.004409194247092962, 'RMSE': np.float64(3.436379188516993), 'Time taken': 0.18559551239013672}
{'Model': 'ElasticNet', 'R-Squared': 0.6813941585606564, 'Adjusted R-Squared': 0.6806102233820177, 'RM

  0%|          | 0/42 [00:00<?, ?it/s]

{'Model': 'AdaBoostRegressor', 'R-Squared': 0.5033949181802528, 'Adjusted R-Squared': 0.5021730127946813, 'RMSE': np.float64(2.4192734448165645), 'Time taken': 28.19593906402588}
{'Model': 'BaggingRegressor', 'R-Squared': 0.9192230656413799, 'Adjusted R-Squared': 0.919024312598497, 'RMSE': np.float64(0.9757151809157253), 'Time taken': 30.52344560623169}
{'Model': 'BayesianRidge', 'R-Squared': 0.868079190568584, 'Adjusted R-Squared': 0.8677545971372753, 'RMSE': np.float64(1.2469126542734812), 'Time taken': 0.47280335426330566}
{'Model': 'DecisionTreeRegressor', 'R-Squared': 0.8552884799309642, 'Adjusted R-Squared': 0.8549324147350974, 'RMSE': np.float64(1.3059631761085755), 'Time taken': 4.383049726486206}
{'Model': 'DummyRegressor', 'R-Squared': -0.0019438939815170642, 'Adjusted R-Squared': -0.004409194247092962, 'RMSE': np.float64(3.436379188516993), 'Time taken': 0.1589198112487793}
{'Model': 'ElasticNet', 'R-Squared': 0.681394158560695, 'Adjusted R-Squared': 0.6806102233820562, 'RMS

,r2_score,MAE,RMSE,MAE_relative(%),scaling_mode,temps_execution
NuSVR,0.93,0.44,0.90,17.05,scaled,815.51
NuSVR,0.93,0.44,0.90,17.05,unscaled,835.18
SVR,0.93,0.44,0.90,17.34,unscaled,239.31
SVR,0.93,0.44,0.90,17.34,scaled,234.03
MLPRegressor,0.93,0.49,0.92,19.37,scaled,72.25
MLPRegressor,0.93,0.49,0.92,19.37,unscaled,71.60
ExtraTreesRegressor,0.92,0.44,0.94,17.30,unscaled,69.88
ExtraTreesRegressor,0.92,0.44,0.94,17.30,scaled,74.53
HistGradientBoostingRegressor,0.92,0.46,0.94,17.83,unscaled,6.65
LGBMRegressor,0.92,0.46,0.94,17.86,unscaled,1.57


In [ ]:
# Sauvegarde les résultats de LazyPredict dans un fichier CSV
# pour éviter de relancer l'entraînement (qui prend du temps)
models_df.reset_index(names="Model").to_csv("models_lazy_predict.csv", index=False)

# models_df.loc[models_df["r2_score"] > 0].sort_values(["r2_score", "MAE"], ascending=[False, True])
models_df.reset_index(names="Model").set_index('Model').sort_values(by=["MAE", "r2_score", "Model"], ascending=[True, False, True]).to_csv("models_lazy_predict.csv")

In [28]:

# -----------------------------
# Split temporel train/test
# -----------------------------
train = df.loc[df['year'] <= 2023].copy()
test = df.loc[df['year'] == 2024].copy()
    
# Matrices train/test
X_train = train[features]
y_train = train["target_variability"]

X_test = test[features]
y_test = test["target_variability"]

# -----------------------------
# Séparation colonnes numériques / catégorielles
# -----------------------------
num_cols = X_train.select_dtypes(include=["number"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

# -----------------------------
# Préprocessing: unscaled vs scaled
# -----------------------------
numeric_scale = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# A) Préprocesseur sans normalisation :
# - Les variables numériques sont conservées telles quelles (pas de scaling)
# - Les variables catégorielles sont transformées en variables binaires (One-Hot Encoding)
preprocess_unscaled = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

# B) Préprocesseur avec normalisation :
# - Les variables numériques sont standardisées (StandardScaler)
# - Les variables catégorielles sont encodées en variables binaires (One-Hot Encoding)
preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_scale, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

In [ ]:
# ---------------------------------
# LazyRegressor + MAE + MAE relatif
# ---------------------------------
def run_lazy(preprocess, label):
    # Fit/transform train, transform test
    Xtr = preprocess.fit_transform(X_train)
    Xte = preprocess.transform(X_test)

    # Lancer LazyRegressor + récupérer prédictions
    reg = LazyRegressor(verbose=1, ignore_warnings=True, random_state=42, predictions=True)
    models, preds = reg.fit(Xtr, Xte, y_train, y_test)
    
    # Référence pour MAE relative (%)
    y_mean = float(np.mean(y_test))

    mae_scores = {}
    mae_rel_scores = {}

    for model_name in preds.columns:
        pred = preds[model_name]
        mae = mean_absolute_error(y_test, pred)
        mae_rel = round(mae / y_mean * 100, 2) if y_mean != 0 else np.nan

        mae_scores[model_name] = mae
        mae_rel_scores[model_name] = mae_rel

    # Fusionner scores MAE avec tableau LazyPredict
        mae_df = pd.DataFrame({
        "MAE": pd.Series(mae_scores),
        "MAE_relatif_%": pd.Series(mae_rel_scores)
    })

    out = models.merge(mae_df, left_index=True, right_index=True)
    out["scaling_mode"] = label
    return out

# Runs
results_unscaled_2024 = run_lazy(preprocess_unscaled, "unscaled")
results_scaled_2024   = run_lazy(preprocess_scaled, "scaled")

# Combiner les deux modes
results_all_2024 = pd.concat([results_unscaled_2024, results_scaled_2024], axis=0)

# Tableau final propre
models_df_2024 = pd.DataFrame(results_all_2024)
models_df_2024 = models_df_2024.rename({
    "R-Squared":"r2_score", 
    "MAE_relatif_%":"MAE_relative(%)", 
    "Time Taken":"temps_execution"
}, axis=1)
models_df_2024 = models_df_2024[["r2_score", "MAE", "RMSE", "MAE_relative(%)", "scaling_mode", "temps_execution"]]

# afficher les 40 meilleurs modèles avec R² > 0, trier par R² décroissant puis MAE croissant
models_df_2024.loc[models_df_2024["r2_score"] > 0].sort_values(["r2_score", "MAE"], ascending=[False, True]).head(40)

In [34]:
models_df_2024.loc[models_df_2024["r2_score"] > 0].sort_values(["r2_score", "MAE"], ascending=[False, True]).head(40)

,r2_score,MAE,RMSE,MAE_relative(%),scaling_mode,temps_execution
SVR,0.90,0.46,1.05,19.01,scaled,505.27
SVR,0.90,0.46,1.05,19.01,unscaled,441.35
NuSVR,0.90,0.45,1.05,18.74,unscaled,2084.54
NuSVR,0.90,0.45,1.05,18.74,scaled,2022.16
HistGradientBoostingRegressor,0.90,0.47,1.06,19.52,scaled,2.49
ExtraTreesRegressor,0.89,0.46,1.07,18.95,unscaled,110.95
ExtraTreesRegressor,0.89,0.46,1.07,18.95,scaled,110.08
LGBMRegressor,0.89,0.47,1.07,19.56,unscaled,1.67
HistGradientBoostingRegressor,0.89,0.47,1.07,19.61,unscaled,2.60
LGBMRegressor,0.89,0.47,1.07,19.65,scaled,1.74


In [ ]:
# Sauvegarde les résultats de LazyPredict dans un fichier CSV
# pour éviter de relancer l'entraînement (qui prend du temps)
models_df_2024.reset_index(names="Model").to_csv("models_2024_lazy_predict.csv", index=False)

In [32]:
# models_df.loc[models_df["r2_score"] > 0].sort_values(["r2_score", "MAE"], ascending=[False, True])
models_df_2024.reset_index(names="Model").set_index('Model').sort_values(by=["MAE", "r2_score", "Model"], ascending=[True, False, True]).to_csv("models_2024_lazy_predict.csv")